# Plot per-variable scores by backend LLM

In [ ]:
#
# This notebook supports one or two JSON files with metric labels such as:
#
# - `ALL.f1`
# - `ALL.precision`
# - `ALL.recall`
# - `habitat.f1`
# - `ecosystem_type.category.support`
# - `organism_trends.Antwortvariable.recall`
#
# The final suffix after the last dot is interpreted as the metric.
# Everything before the final dot is interpreted as the variable.
#
# If one experiment is configured, each plot is a simple bar chart.
# If two experiments are configured, each plot is a grouped bar chart comparing the experiments.

import json

# %%
from pathlib import Path
import re

import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import numpy as np

# Optional IEEE/science style.
try:
    import scienceplots  # noqa: F401

    # plt.style.use(["science", "ieee", "no-latex"])
except Exception:
    print("scienceplots not available; using current matplotlib style.")

## Configuration

In [ ]:
# %%
# Configure SCHEMA, EXPERIMENTS, EXPERIMENT_NAME, SERIES_NAME_MAP and SORT_BY at least! Optionally: EXPERIMENT_ORDER
SCHEMA = "Organism Trends (biodiversity variable and trend given habitat and species group)"
# used as part of the filename to create unique file names per plotting goal. Use sth to describe
# the plot so that it can be identified reliably, in conjunction with the SCHEMA, VARIABLE, and METRIC
# Final filenames look sth like {SCHEMA}_{EXPERIMENT_FILENAME}_{VARIABLE}_{METRIC}.pdf/png
# Examples: 'test' or 'dev_vs_test'
EXPERIMENT_FILENAME = "test"
# Input path can be one combined file with multiple series
EXPERIMENTS = [
    {
        "path": Path(
            "../../kibad-llm-results/logs/549_organism_trends_bestconfig_testset/figure_data/organism_trends_f1_micro_conditional_variable_and_trend-ALL-data.json"
        ),
    },
]

# Or configure either one experiment...
# EXPERIMENTS = [
#    {
#        "name": "Test",
#        "path": Path(
#            "../data/prediction_results/logs/549_organism_trends_bestconfig_testset/"
#            "figure_data/organism_trends_f1_micro_conditional_variable_and_trend-ALL-data.json"
#        ),
#    },
# ]

# ...or two experiments.
# EXPERIMENTS = [
#     {
#         "name": "Dev (corrected annotations)",
#         "path": Path(
#             "../data/prediction_results/logs/519_faktencheck_core/"
#             "figure_data/faktencheck_core_f1_micro_flat-ALL-data.json"
#         ),
#     },
#     {
#         "name": "Faktencheck Core test set (uncorrected annotations)",
#         "path": Path(
#             "../data/prediction_results/logs/525_faktencheck_core_bestconfig_testset/"
#             "figure_data/faktencheck_core_f1_micro_flat-ALL-data.json"
#         ),
#     },
# ]

# Optional mapping from raw JSON series names to nicer experiment names.
#
# In combined files, series labels often look like:
#   name=525_faktencheck_core_bestconfig_testset
#
# The loader strips the leading "name=" automatically, so the keys below should
# usually not include "name=".
SERIES_NAME_MAP = {
    "549_organism_trends_bestconfig_testset": "Test",
    # "519_faktencheck_core": "Dev",
}

# Optional explicit experiment order after loading.
# Use the display names after SERIES_NAME_MAP has been applied.
#
# If None, experiments keep the order in which they are encountered.
EXPERIMENT_ORDER = None
# EXPERIMENT_ORDER = [
#     "Dev",
#     "Test",
# ]

# Sort model bars/groups by a reference experiment, variable, and metric.
# Set to None to keep the order from the JSON file.
#
# For one experiment, "experiment" may be omitted or set to None.
SORT_BY = {
    "experiment": None,
    "variable": None,  # or None
    "metric": "f1",  # or None
}


OUTPUT_DIR = Path("plots")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FORMATS = ["png", "pdf"]

# If None, all metrics found in the JSON are plotted.
# Otherwise, use e.g. ["f1", "precision", "recall"].
METRICS_TO_INCLUDE = None
# METRICS_TO_INCLUDE = ["f1", "precision", "recall"]

# If None, all variables found in the JSON are plotted.
# Otherwise, use e.g. ["ALL", "AVG", "habitat"].
VARIABLES_TO_INCLUDE = None
# VARIABLES_TO_INCLUDE = ["ALL", "AVG", "biodiversity_level", "habitat"]

# Error bars:
# "std" = standard deviation
# "sem" = standard error of the mean
ERRORBAR = "std"

# Figure sizing.
IEEE_ONE_COLUMN_WIDTH_IN = 7
FIG_HEIGHT_IN = 5.4

# Width of a single bar in one-experiment plots.
BAR_WIDTH = 0.55

# Total width occupied by all experiment bars inside each model group.
# Only relevant when len(EXPERIMENTS) > 1.
GROUP_WIDTH = 0.62

# If True, each plot is sorted by its own metric values.
# If False, all plots use the same model order.
SORT_EACH_PLOT_INDIVIDUALLY = False


# If True, use shortened multi-line model labels on the x-axis.
USE_SHORT_MODEL_LABELS = False

# %% [markdown]
# ## Display names and colors

# %%
PRETTY_METRIC = {
    "f1": "F1",
    "precision": "Precision",
    "recall": "Recall",
    "support": "Support",
}

PRETTY_LLM = {
    "gpt_oss_20b": "GPT OSS 20B",
    "mistral_small_3_24b": "Mistral Small 3 24B",
    "gemma3_27b": "Gemma 3 27B",
    "qwen3_30b": "Qwen 3 30B",
    "gpt_5": "GPT-5",
}

PRETTY_VARIABLE = {
    "ALL": "Micro average",
    "AVG": "Macro average",
    "biodiversity_level": "Biodiversity level",
    "ecosystem_type.category": "Ecosystem type - category",
    "ecosystem_type.term": "Ecosystem type - term",
    "habitat": "Habitat",
    "taxa.species_group": "Taxa - species group",
    "Lebensraum": "Habitat",
    "Antwortvariable": "Biodiversity variable",
    "Hauptgruppe_RoteListen": "Main species group red lists",
    "Trend": "Trend",
}

# Colorblind-friendly Okabe-Ito-inspired palette.
MODEL_COLORS = [
    "#0072B2",  # blue
    "#E69F00",  # orange
    "#009E73",  # green
    "#CC79A7",  # purple
    "#D55E00",  # vermillion
    "#56B4E9",  # sky blue
    "#F0E442",  # yellow
]

## Helper functions

In [ ]:
# %%


def clean_category_label(category: str) -> str:
    """
    Convert backend category strings into readable model names.

    Example
    -------
    prediction.extractor/llm=gpt_oss_20b_in_process
    -> GPT OSS 20B
    """
    match = re.search(r"llm=([^/]+)", category)
    label = match.group(1) if match else category

    if label.endswith("_in_process"):
        label = label[: -len("_in_process")]

    return PRETTY_LLM.get(label, label.replace("_", " ").title())


def short_model_label(label: str) -> str:
    """
    Return a shortened/multi-line model label for compact plots.
    """
    replacements = {
        "GPT OSS 20B": "GPT OSS\n20B",
        "Mistral Small 3 24B": "Mistral\nSmall 3",
        "Gemma 3 27B": "Gemma 3\n27B",
        "Qwen 3 30B": "Qwen 3\n30B",
        "GPT-5": "GPT-5",
    }
    return replacements.get(label, label)


def clean_series_label(series: str | None, fallback_name: str | None = None) -> str:
    """
    Convert a JSON point `series` value into an experiment name.

    Examples
    --------
    "__single__" -> fallback_name
    "name=525_faktencheck_core_bestconfig_testset"
        -> "525_faktencheck_core_bestconfig_testset"
        -> SERIES_NAME_MAP value if configured

    Parameters
    ----------
    series:
        Raw `point["series"]` or `point["display_series"]` value.
    fallback_name:
        Name to use for single-experiment files where the series is "__single__".

    Returns
    -------
    str
        Clean experiment name.
    """
    if series is None or series == "__single__":
        if fallback_name is None:
            return "__single__"
        return fallback_name

    # Common format in combined files.
    if series.startswith("name="):
        series = series[len("name=") :]

    return SERIES_NAME_MAP.get(series, series)


def format_model_label(category: str) -> str:
    """
    Format a model/category label for the x-axis.
    """
    label = clean_category_label(category)
    return short_model_label(label) if USE_SHORT_MODEL_LABELS else label


def pretty_variable_name(variable: str) -> str:
    return PRETTY_VARIABLE.get(variable, variable.replace("_", " "))


def pretty_metric_name(metric: str) -> str:
    return PRETTY_METRIC.get(metric, metric.replace("_", " ").title())


def safe_filename(text: str) -> str:
    """
    Convert arbitrary text into a filesystem-friendly filename component.
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9._-]+", "_", text)
    text = text.replace(".", "_")
    return text.strip("_")


def split_metric_label(metric_label: str) -> tuple[str, str]:
    """
    Split metric labels of the form:

        variable.metric

    where the variable itself may contain dots.

    Examples
    --------
    ALL.f1 -> variable='ALL', metric='f1'
    habitat.precision -> variable='habitat', metric='precision'
    ecosystem_type.category.f1 -> variable='ecosystem_type.category', metric='f1'
    """
    if "." not in metric_label:
        raise ValueError(
            f"Expected metric label of form '<variable>.<metric>', got: {metric_label!r}"
        )

    variable, metric = metric_label.rsplit(".", 1)
    return variable, metric


def mean_and_error(samples, errorbar: str = "std") -> tuple[float, float, int]:
    """
    Compute mean and error bar size for one list of samples.

    Parameters
    ----------
    samples:
        Numeric sample values.
    errorbar:
        Either "std" or "sem".

    Returns
    -------
    mean, err, n
    """
    samples = np.asarray(samples, dtype=float)

    if len(samples) == 0:
        raise ValueError("Found a point with an empty `samples` list.")

    mean = float(np.mean(samples))

    if len(samples) == 1:
        return mean, 0.0, 1

    std = float(np.std(samples, ddof=1))

    if errorbar == "std":
        err = std
    elif errorbar == "sem":
        err = std / np.sqrt(len(samples))
    else:
        raise ValueError(f"Unknown errorbar type: {errorbar!r}")

    return mean, float(err), int(len(samples))


def blend_with(color, target_color, amount: float):
    """
    Blend a color with another target color.

    amount=0 returns the original color.
    amount=1 returns the target color.
    """
    color = np.array(mcolors.to_rgb(color))
    target = np.array(mcolors.to_rgb(target_color))
    return tuple((1 - amount) * color + amount * target)


def lighten_color(color, amount: float = 0.45):
    return blend_with(color, "white", amount)


def darken_color(color, amount: float = 0.15):
    return blend_with(color, "black", amount)


def color_for_experiment(
    category: str, model_base_colors: dict[str, str], exp_index: int, n_experiments: int
):
    """
    Return the bar color for a category/experiment combination.

    For one experiment:
    - use the model's base color.

    For two experiments:
    - experiment 0 uses a light shade of the model color;
    - experiment 1 uses a dark shade of the model color.
    """
    base = model_base_colors[category]

    if n_experiments == 1:
        return base

    if n_experiments == 2:
        return (
            lighten_color(base, amount=0.45) if exp_index == 0 else darken_color(base, amount=0.15)
        )

    # Fallback for more than two experiments.
    # The notebook is designed for one or two, but this avoids hard failure.
    return base

## Loading and validation

In [ ]:
# %%


def load_experiment_file(
    json_path: Path,
    experiment_name: str | None = None,
    metrics_to_include=None,
    variables_to_include=None,
    errorbar: str = "std",
):
    """
    Load one JSON file.

    This function supports both JSON layouts:

    1. Single-experiment files
       All points usually have:
           point["series"] == "__single__"

       In that case the returned dictionary contains one experiment, named by
       `experiment_name` if provided, otherwise by the file stem.

    2. Combined multi-experiment files
       Points contain different `series` values, for example:
           "name=525_faktencheck_core_bestconfig_testset"
           "name=519_faktencheck_core"

       In that case each distinct series is treated as a separate experiment.

    Returns
    -------
    experiments:
        Dictionary:

            experiments[experiment_name] = {
                "categories": [...],
                "stats": stats,
                "pair_order": [...]
            }

        where:

            stats[metric][variable][category] = {
                "mean": float,
                "err": float,
                "n": int,
                "samples": list,
            }
    """
    json_path = Path(json_path)
    fallback_name = experiment_name or json_path.stem

    with json_path.open("r", encoding="utf-8") as f:
        obj = json.load(f)

    loaded_experiments = {}

    def ensure_experiment_exists(name: str):
        if name not in loaded_experiments:
            loaded_experiments[name] = {
                "categories": [],
                "stats": {},
                "pair_order": [],
            }

    for plot in obj.get("plots", []):
        metric_label = plot["metadata"]["metric_label"]
        variable, metric = split_metric_label(metric_label)

        if metrics_to_include is not None and metric not in metrics_to_include:
            continue

        if variables_to_include is not None and variable not in variables_to_include:
            continue

        pair = (metric, variable)
        points = plot["data"]["points"]

        for point in points:
            # Prefer display_series if present, otherwise use series.
            raw_series = point.get("display_series", point.get("series", "__single__"))
            exp_name = clean_series_label(raw_series, fallback_name=fallback_name)

            ensure_experiment_exists(exp_name)

            exp_data = loaded_experiments[exp_name]
            categories = exp_data["categories"]
            stats = exp_data["stats"]
            pair_order = exp_data["pair_order"]

            category = point["category"]
            samples = point["samples"]

            if category not in categories:
                categories.append(category)

            if pair not in pair_order:
                pair_order.append(pair)

            stats.setdefault(metric, {})
            stats[metric].setdefault(variable, {})

            if category in stats[metric][variable]:
                raise ValueError(
                    f"Duplicate point found in {json_path} for "
                    f"experiment={exp_name!r}, variable={variable!r}, "
                    f"metric={metric!r}, category={category!r}. "
                    "This usually means the file contains duplicate series/category "
                    "entries for the same metric label."
                )

            mean, err, n = mean_and_error(samples, errorbar=errorbar)

            stats[metric][variable][category] = {
                "mean": mean,
                "err": err,
                "n": n,
                "samples": samples,
            }

    if not loaded_experiments:
        raise ValueError(f"No matching plots found in {json_path}")

    # Validate category consistency inside each loaded experiment.
    for exp_name, exp_data in loaded_experiments.items():
        categories = exp_data["categories"]
        stats = exp_data["stats"]

        if not categories:
            raise ValueError(f"No categories found for experiment {exp_name!r} in {json_path}")

        for metric, variable_dict in stats.items():
            for variable, category_dict in variable_dict.items():
                missing_categories = [
                    category for category in categories if category not in category_dict
                ]

                if missing_categories:
                    raise ValueError(
                        f"In file {json_path}, experiment={exp_name!r}, "
                        f"metric={metric!r}, variable={variable!r} is missing "
                        f"categories: {missing_categories}"
                    )

    return loaded_experiments


def load_experiments(experiments):
    """
    Load all configured experiment files.

    Each configured file may contain either:

    - one experiment, encoded with series="__single__";
    - multiple experiments, encoded by distinct point["series"] values.

    Returns
    -------
    dict with keys:
        experiment_stats
        experiment_categories
        experiment_pair_orders
        ordered_pairs
        categories
        reference_experiment
        experiment_names
    """
    if not experiments:
        raise ValueError("Configure at least one experiment in EXPERIMENTS.")

    experiment_stats = {}
    experiment_categories = {}
    experiment_pair_orders = {}
    experiment_names = []

    for exp in experiments:
        path = exp["path"]
        configured_name = exp.get("name")

        loaded_from_file = load_experiment_file(
            path,
            experiment_name=configured_name,
            metrics_to_include=METRICS_TO_INCLUDE,
            variables_to_include=VARIABLES_TO_INCLUDE,
            errorbar=ERRORBAR,
        )

        for name, payload in loaded_from_file.items():
            if name in experiment_stats:
                raise ValueError(
                    f"Duplicate experiment name after loading: {name!r}. "
                    "Use unique EXPERIMENTS names or adjust SERIES_NAME_MAP."
                )

            experiment_names.append(name)
            experiment_categories[name] = payload["categories"]
            experiment_stats[name] = payload["stats"]
            experiment_pair_orders[name] = payload["pair_order"]

    if EXPERIMENT_ORDER is not None:
        missing = set(EXPERIMENT_ORDER) - set(experiment_names)
        extra = set(experiment_names) - set(EXPERIMENT_ORDER)

        if missing or extra:
            raise ValueError(
                "EXPERIMENT_ORDER does not match loaded experiments.\n"
                f"Missing from loaded experiments: {missing}\n"
                f"Loaded but not listed in EXPERIMENT_ORDER: {extra}"
            )

        experiment_names = list(EXPERIMENT_ORDER)

    reference_experiment = experiment_names[0]
    reference_categories = list(experiment_categories[reference_experiment])
    reference_category_set = set(reference_categories)

    # Ensure all experiments contain the same model categories.
    for name in experiment_names[1:]:
        category_set = set(experiment_categories[name])

        if category_set != reference_category_set:
            missing = reference_category_set - category_set
            extra = category_set - reference_category_set
            raise ValueError(
                f"Model categories differ for experiment {name!r}.\n"
                f"Missing from {name!r}: {missing}\n"
                f"Extra in {name!r}: {extra}"
            )

    # Keep only metric-variable pairs present in all experiments.
    pair_sets = []

    for name in experiment_names:
        pairs = {
            (metric, variable)
            for metric in experiment_stats[name]
            for variable in experiment_stats[name][metric]
        }
        pair_sets.append(pairs)

    common_pairs = set.intersection(*pair_sets)

    # Preserve metric-variable order from the reference experiment.
    ordered_pairs = [
        pair for pair in experiment_pair_orders[reference_experiment] if pair in common_pairs
    ]

    if not ordered_pairs:
        raise ValueError("No common metric-variable pairs found across experiments.")

    return {
        "experiment_stats": experiment_stats,
        "experiment_categories": experiment_categories,
        "experiment_pair_orders": experiment_pair_orders,
        "ordered_pairs": ordered_pairs,
        "categories": reference_categories,
        "reference_experiment": reference_experiment,
        "experiment_names": experiment_names,
    }

## Load data

In [ ]:
# %%
loaded = load_experiments(EXPERIMENTS)

experiment_stats = loaded["experiment_stats"]
experiment_categories = loaded["experiment_categories"]
ordered_pairs = loaded["ordered_pairs"]
categories = loaded["categories"]
reference_experiment = loaded["reference_experiment"]
experiment_names = loaded["experiment_names"]

n_experiments = len(experiment_names)

metrics = list(dict.fromkeys(metric for metric, _variable in ordered_pairs))
variables = list(dict.fromkeys(variable for _metric, variable in ordered_pairs))

print("Loaded experiments:")
for name in experiment_names:
    print(f"  {name}")

print("\nCommon metric-variable pairs:")
for metric, variable in ordered_pairs:
    print(f"  {variable}.{metric}")

print("\nModels:")
for category in categories:
    print(f"  {clean_category_label(category)}")

## Sort models

In [ ]:
# %%


def resolve_sort_reference(
    sort_by: dict,
    experiment_names: list[str],
    ordered_pairs: list[tuple[str, str]],
) -> tuple[str, str, str]:
    """
    Resolve a possibly partially specified SORT_BY dictionary.

    Rules
    -----
    - experiment=None -> use the first loaded experiment.
    - metric=None and variable=None -> use the first metric-variable pair.
    - metric=None and variable is set -> use the first metric available for that variable.
    - variable=None and metric is set -> use the first variable available for that metric.
    - both metric and variable are set -> use them directly.

    Returns
    -------
    sort_experiment, sort_metric, sort_variable
    """
    if not experiment_names:
        raise ValueError("No experiments were loaded.")

    if not ordered_pairs:
        raise ValueError("No metric-variable pairs were loaded.")

    sort_experiment = sort_by.get("experiment") or experiment_names[0]
    sort_metric = sort_by.get("metric")
    sort_variable = sort_by.get("variable")

    if sort_metric is None and sort_variable is None:
        sort_metric, sort_variable = ordered_pairs[0]

    elif sort_metric is None:
        matching_pairs = [
            (metric, variable) for metric, variable in ordered_pairs if variable == sort_variable
        ]

        if not matching_pairs:
            raise ValueError(
                f"SORT_BY requested variable={sort_variable!r}, but this variable "
                "does not occur in ordered_pairs."
            )

        sort_metric, sort_variable = matching_pairs[0]

    elif sort_variable is None:
        matching_pairs = [
            (metric, variable) for metric, variable in ordered_pairs if metric == sort_metric
        ]

        if not matching_pairs:
            raise ValueError(
                f"SORT_BY requested metric={sort_metric!r}, but this metric "
                "does not occur in ordered_pairs."
            )

        sort_metric, sort_variable = matching_pairs[0]

    return sort_experiment, sort_metric, sort_variable


def sort_categories(
    categories: list[str],
    experiment_stats: dict,
    experiment_names: list[str],
    ordered_pairs: list[tuple[str, str]],
    sort_by: dict | None,
) -> list[str]:
    """
    Sort model categories according to SORT_BY.

    If sort_by is None, the original JSON/category order is kept.

    Otherwise, SORT_BY may contain None values:

        SORT_BY = {
            "experiment": None,
            "variable": None,
            "metric": None,
        }

    In that case, the first available experiment and/or metric-variable pair
    is used as the sorting reference.
    """
    if sort_by is None:
        print("Keeping original ordering")
        return list(categories)

    sort_experiment, sort_metric, sort_variable = resolve_sort_reference(
        sort_by=sort_by,
        experiment_names=experiment_names,
        ordered_pairs=ordered_pairs,
    )

    if sort_experiment not in experiment_stats:
        raise ValueError(
            f"Sort experiment {sort_experiment!r} not found. "
            f"Available experiments: {list(experiment_stats.keys())}"
        )

    if sort_metric not in experiment_stats[sort_experiment]:
        raise ValueError(
            f"Sort metric {sort_metric!r} not found in experiment "
            f"{sort_experiment!r}. Available metrics: "
            f"{list(experiment_stats[sort_experiment].keys())}"
        )

    if sort_variable not in experiment_stats[sort_experiment][sort_metric]:
        raise ValueError(
            f"Sort variable {sort_variable!r} not found for "
            f"experiment={sort_experiment!r}, metric={sort_metric!r}. "
            f"Available variables: "
            f"{list(experiment_stats[sort_experiment][sort_metric].keys())}"
        )

    SORT_BY["experiment"] = sort_experiment
    SORT_BY["metric"] = sort_metric
    SORT_BY["variable"] = sort_variable

    print("Sorting experiments by:")
    print(f"  experiment: {sort_experiment}")
    print(f"  variable: {sort_variable}")
    print(f"  metric: {sort_metric}")

    return sorted(
        categories,
        key=lambda category: experiment_stats[sort_experiment][sort_metric][sort_variable][
            category
        ]["mean"],
        reverse=True,
    )


categories = sort_categories(
    categories=categories,
    experiment_stats=experiment_stats,
    experiment_names=experiment_names,
    ordered_pairs=ordered_pairs,
    sort_by=SORT_BY,
)

# Assign stable model colors after sorting.
model_colors = {
    category: MODEL_COLORS[i % len(MODEL_COLORS)] for i, category in enumerate(categories)
}

print("Model order:")
for category in categories:
    label = clean_category_label(category)

    if SORT_BY is not None:
        value = experiment_stats[SORT_BY["experiment"]][SORT_BY["metric"]][SORT_BY["variable"]][
            category
        ]["mean"]

        print(
            f"  {label}: {value:.4f} "
            f"({SORT_BY["experiment"]}, {SORT_BY["variable"]}.{SORT_BY["metric"]})"
        )
    else:
        print(f"  {label}")

## Unified plot function

In [ ]:
#
# This single function handles both:
#
# - one experiment: ordinary bar chart;
# - two experiments: grouped bar chart.


# %%
def get_plot_categories(metric: str, variable: str) -> list[str]:
    """
    Return category order for a specific plot.

    If SORT_EACH_PLOT_INDIVIDUALLY is True, categories are sorted by the first
    experiment's values for this plot. Otherwise, the global category order is used.
    """
    if not SORT_EACH_PLOT_INDIVIDUALLY:
        return list(categories)

    sort_experiment = experiment_names[0]

    return sorted(
        categories,
        key=lambda category: experiment_stats[sort_experiment][metric][variable][category]["mean"],
        reverse=True,
    )


def validate_metric_variable_available(metric: str, variable: str):
    """
    Ensure that the requested metric-variable pair exists in every experiment.
    """
    for exp_name in experiment_names:
        if metric not in experiment_stats[exp_name]:
            raise ValueError(f"Metric {metric!r} missing in experiment {exp_name!r}")

        if variable not in experiment_stats[exp_name][metric]:
            raise ValueError(
                f"Variable {variable!r} missing for metric {metric!r} "
                f"in experiment {exp_name!r}"
            )


def format_score(value: float) -> str:
    """
    Format bar labels.

    Scores in [0, 1] are shown with 3 significant digits.
    Larger values, for example support counts, are shown without decimals.
    """
    if value <= 1:
        return f"{value:.3g}"
    return f"{value:.0f}"


def set_y_limits(ax, metric: str, ymax: float, n_experiments: int):
    """
    Set y-axis limits.

    Most metrics are expected to be in [0, 1].
    Support-like metrics may be larger.
    """
    if metric.lower() == "support":
        multiplier = 1.55 if n_experiments > 1 else 1.15
        ax.set_ylim(0, ymax * multiplier)
    else:
        ax.set_ylim(0, 1.05)


def add_experiment_legend(ax):
    """
    Add a legend explaining the experiment shade encoding.

    Only used for two-experiment plots.
    """
    if n_experiments != 2:
        return

    legend_handles = [
        Patch(
            facecolor="0.80",
            edgecolor="black",
            linewidth=0.5,
            label=f"{experiment_names[0]}: light shade",
        ),
        Patch(
            facecolor="0.35",
            edgecolor="black",
            linewidth=0.5,
            label=f"{experiment_names[1]}: dark shade",
        ),
    ]

    ax.legend(
        handles=legend_handles,
        fontsize=10,
        frameon=True,
        loc="upper right",
    )


def plot_variable_metric(metric: str, variable: str, output_paths):
    """
    Create and save one plot for one variable-metric pair.

    The function automatically adapts to the number of configured experiments:

    - one experiment: simple bar chart;
    - two experiments: grouped bar chart.

    Parameters
    ----------
    metric:
        Metric suffix, e.g. "f1", "precision", "recall".
    variable:
        Variable part of the metric label, e.g. "ALL", "habitat",
        "ecosystem_type.category".
    output_paths:
        Iterable of output paths, e.g. PNG and PDF paths.

    Returns
    -------
    fig, ax
    """
    validate_metric_variable_available(metric, variable)

    if n_experiments > 2:
        print(
            "Warning: this notebook is designed for one or two experiments. "
            "Additional experiments will reuse the base model color."
        )

    plot_categories = get_plot_categories(metric, variable)
    x = np.arange(len(plot_categories))

    if n_experiments == 1:
        bar_width = BAR_WIDTH
        offsets = np.array([0.0])
    else:
        bar_width = GROUP_WIDTH / n_experiments
        offsets = (np.arange(n_experiments) - (n_experiments - 1) / 2) * bar_width

    fig, ax = plt.subplots(figsize=(IEEE_ONE_COLUMN_WIDTH_IN, FIG_HEIGHT_IN))

    all_ymax_candidates = []

    for exp_index, exp_name in enumerate(experiment_names):
        means = np.array(
            [
                experiment_stats[exp_name][metric][variable][category]["mean"]
                for category in plot_categories
            ]
        )
        errs = np.array(
            [
                experiment_stats[exp_name][metric][variable][category]["err"]
                for category in plot_categories
            ]
        )
        ns = np.array(
            [
                experiment_stats[exp_name][metric][variable][category]["n"]
                for category in plot_categories
            ]
        )

        bar_positions = x + offsets[exp_index]

        colors = [
            color_for_experiment(
                category=category,
                model_base_colors=model_colors,
                exp_index=exp_index,
                n_experiments=n_experiments,
            )
            for category in plot_categories
        ]

        bars = ax.bar(
            bar_positions,
            means,
            width=bar_width if n_experiments == 1 else bar_width * 0.90,
            color=colors,
            edgecolor="black",
            linewidth=0.6 if n_experiments == 1 else 0.45,
            label=exp_name,
        )

        # Add error bars only when there is more than one sample.
        for xi, yi, err, n in zip(bar_positions, means, errs, ns):
            if n > 1:
                ax.errorbar(
                    xi,
                    yi,
                    yerr=err,
                    fmt="none",
                    ecolor="black",
                    elinewidth=0.8 if n_experiments == 1 else 0.65,
                    capsize=8.5 if n_experiments == 1 else 4.0,
                    capthick=0.8 if n_experiments == 1 else 0.65,
                )

        # Print scores above bars.
        for bar, value, err, n in zip(bars, means, errs, ns):
            y = value + (err if n > 1 else 0.0) + 0.010

            ax.text(
                bar.get_x() + bar.get_width() / 2,
                y if n_experiments == 1 else y * 1.03,
                format_score(value),
                ha="center",
                va="bottom",
                fontsize=10,
                rotation=0 if n_experiments == 1 else 90,
            )

        all_ymax_candidates.extend(means + np.where(ns > 1, errs, 0.0))

    title = f"{pretty_variable_name(variable)} {pretty_metric_name(metric)}"

    if n_experiments == 1:
        title = f"{title} by backend LLM on the {SCHEMA} schema"

    ax.set_title(
        title,
        fontsize=10,
        fontweight="bold",
        pad=10,
    )

    ax.set_ylabel(pretty_metric_name(metric), fontsize=10)

    ax.set_xticks(x)
    ax.set_xticklabels(
        [format_model_label(category) for category in plot_categories],
        rotation=45,
        ha="center",
        fontsize=10,
    )

    ax.tick_params(axis="y", labelsize=10)
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax.set_axisbelow(True)

    ymax = float(np.max(all_ymax_candidates))
    set_y_limits(ax, metric=metric, ymax=ymax, n_experiments=n_experiments)

    ax.set_xlim(-0.4, len(plot_categories) - 0.6)

    if n_experiments > 1:
        add_experiment_legend(ax)

    fig.tight_layout(pad=0.4 if n_experiments == 1 else 0.35)

    for output_path in output_paths:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        fig.savefig(
            output_path,
            dpi=300,
            bbox_inches="tight",
        )

        print(f"Saved {variable}.{metric} plot to: {output_path.resolve()}")

    plt.show()

    return fig, ax

## Generate plots

In [ ]:
# %%
created_figures = {}

for metric, variable in ordered_pairs:
    filename_parts = [
        safe_filename(SCHEMA),
        safe_filename(EXPERIMENT_FILENAME),
        safe_filename(variable),
        safe_filename(metric),
    ]

    if n_experiments > 1:
        filename_parts.append("grouped_experiments")

    filename_base = "_".join(filename_parts)

    output_paths = [OUTPUT_DIR / f"{filename_base}.{ext}" for ext in OUTPUT_FORMATS]

    fig, ax = plot_variable_metric(
        metric=metric,
        variable=variable,
        output_paths=output_paths,
    )

    created_figures[(metric, variable)] = (fig, ax)